In [ ]:
  from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os

DATASET_PATH = "/content/drive/MyDrive/EEG_Drowsiness_New/Dataset/dataset.mat"

print("Dataset exists:", os.path.exists(DATASET_PATH))

Dataset exists: True


In [ ]:
import scipy.io as sio

mat_data = sio.loadmat(DATASET_PATH)

print(mat_data.keys())

dict_keys(['__header__', '__version__', '__globals__', 'EEGsample', 'subindex', 'substate'])


In [ ]:
import numpy as np

for name in ["EEGsample", "subindex", "substate"]:
    value = mat_data[name]

    print("\n" + "=" * 60)
    print(name)
    print("Python type:", type(value))
    print("Shape:", getattr(value, "shape", None))
    print("Dtype:", getattr(value, "dtype", None))

    if isinstance(value, np.ndarray):
        print("First values:")
        print(value.reshape(-1)[:10])


EEGsample
Python type: <class 'numpy.ndarray'>
Shape: (2022, 30, 384)
Dtype: float64
First values:
[ 4.08097622  5.36373657  2.16892286  6.68276147  9.79531389  8.11839397
  8.33377608  1.47171434 -6.61412784 -6.246151  ]

subindex
Python type: <class 'numpy.ndarray'>
Shape: (2022, 1)
Dtype: uint8
First values:
[1 1 1 1 1 1 1 1 1 1]

substate
Python type: <class 'numpy.ndarray'>
Shape: (2022, 1)
Dtype: uint8
First values:
[0 0 0 0 0 0 0 0 0 0]


In [ ]:
EEGsample = mat_data["EEGsample"]
subindex = mat_data["subindex"]
substate = mat_data["substate"]

print("EEGsample shape:", EEGsample.shape)
print("subindex shape:", subindex.shape)
print("substate shape:", substate.shape)

print("\nEEGsample dtype:", EEGsample.dtype)
print("subindex dtype:", subindex.dtype)
print("substate dtype:", substate.dtype)

EEGsample shape: (2022, 30, 384)
subindex shape: (2022, 1)
substate shape: (2022, 1)

EEGsample dtype: float64
subindex dtype: uint8
substate dtype: uint8


In [ ]:
print("Unique subindex values:")
print(np.unique(subindex))

print("\nUnique substate values:")
print(np.unique(substate))

Unique subindex values:
[ 1  2  3  4  5  6  7  8  9 10 11]

Unique substate values:
[0 1]


In [ ]:
import numpy as np
import pandas as pd

X = EEGsample.astype(np.float32)

# Convert from:
# (samples, channels, time)
# to:
# (samples, time, channels)
X = np.transpose(X, (0, 2, 1))

y = substate.reshape(-1).astype(np.int32)
subjects = subindex.reshape(-1).astype(np.int32)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("subjects shape:", subjects.shape)

X shape: (2022, 384, 30)
y shape: (2022,)
subjects shape: (2022,)


In [ ]:
assert X.shape[0] == len(y)
assert X.shape[0] == len(subjects)

assert set(np.unique(y)).issubset({0, 1})
assert np.isfinite(X).all()

print("Number of samples:", len(X))
print("Number of subjects:", len(np.unique(subjects)))
print("Subjects:", np.unique(subjects))
print("Labels:", np.unique(y))

print("\nOverall label counts:")
print(
    pd.Series(y)
    .value_counts()
    .sort_index()
    .rename(index={
        0: "Alert",
        1: "Drowsy"
    })
)

print("\nSamples per subject:")
print(
    pd.Series(subjects)
    .value_counts()
    .sort_index()
)

Number of samples: 2022
Number of subjects: 11
Subjects: [ 1  2  3  4  5  6  7  8  9 10 11]
Labels: [0 1]

Overall label counts:
Alert     1011
Drowsy    1011
Name: count, dtype: int64

Samples per subject:
1     188
2     132
3     150
4     148
5     224
6     166
7     102
8     264
9     314
10    108
11    226
Name: count, dtype: int64


In [ ]:
subject_label_table = pd.crosstab(
    pd.Series(subjects, name="subject_id"),
    pd.Series(y, name="label")
)

subject_label_table = subject_label_table.rename(
    columns={
        0: "Alert",
        1: "Drowsy"
    }
)

subject_label_table["Total"] = (
    subject_label_table.get("Alert", 0)
    + subject_label_table.get("Drowsy", 0)
)

subject_label_table["Drowsy_ratio"] = (
    subject_label_table.get("Drowsy", 0)
    / subject_label_table["Total"]
)

display(subject_label_table)

label,Alert,Drowsy,Total,Drowsy_ratio
subject_id,,,,
1,94,94,188,0.5
2,66,66,132,0.5
3,75,75,150,0.5
4,74,74,148,0.5
5,112,112,224,0.5
6,83,83,166,0.5
7,51,51,102,0.5
8,132,132,264,0.5
9,157,157,314,0.5


In [ ]:
label_counts = np.bincount(y, minlength=2)

print("Alert samples:", label_counts[0])
print("Drowsy samples:", label_counts[1])

print(
    "Drowsy percentage:",
    100 * label_counts[1] / len(y)
)

Alert samples: 1011
Drowsy samples: 1011
Drowsy percentage: 50.0


In [ ]:
sample_std = X.std(axis=(1, 2))
sample_abs_max = np.max(np.abs(X), axis=(1, 2))

outlier_mask = (
    (sample_std > np.percentile(sample_std, 99))
    | (sample_abs_max > np.percentile(sample_abs_max, 99))
)

print("Total samples:", len(X))
print("Possible extreme samples:", outlier_mask.sum())

outlier_table = pd.DataFrame({
    "sample_index": np.arange(len(X)),
    "subject": subjects,
    "label": y,
    "sample_std": sample_std,
    "absolute_max": sample_abs_max,
    "possible_outlier": outlier_mask
})

display(
    outlier_table
    .sort_values("sample_std", ascending=False)
    .head(20)
)

Total samples: 2022
Possible extreme samples: 22


,sample_index,subject,label,sample_std,absolute_max,possible_outlier
1751,1751,10,1,329.617706,2628.769287,True
1752,1752,10,1,250.505707,2481.432861,True
1744,1744,10,1,247.375488,2872.673584,True
1743,1743,10,1,243.134033,3430.907959,True
1748,1748,10,1,238.395508,2976.987793,True
1750,1750,10,1,206.110260,2805.348389,True
1749,1749,10,1,191.957779,3195.426025,True
1747,1747,10,1,171.252502,2571.461426,True
1742,1742,10,1,169.697510,2662.226318,True
1745,1745,10,1,168.661774,2598.770996,True


In [ ]:
def fit_fold_preprocessor(X_train):
    lower = np.percentile(
        X_train,
        1.0,
        axis=(0, 1),
        keepdims=True
    )

    upper = np.percentile(
        X_train,
        99.0,
        axis=(0, 1),
        keepdims=True
    )

    X_train_clipped = np.clip(
        X_train,
        lower,
        upper
    )

    mean = X_train_clipped.mean(
        axis=(0, 1),
        keepdims=True
    )

    std = X_train_clipped.std(
        axis=(0, 1),
        keepdims=True
    )

    std = np.where(std < 1e-6, 1.0, std)

    return {
        "lower": lower.astype(np.float32),
        "upper": upper.astype(np.float32),
        "mean": mean.astype(np.float32),
        "std": std.astype(np.float32)
    }


def apply_fold_preprocessor(X_data, preprocessor):
    X_clipped = np.clip(
        X_data,
        preprocessor["lower"],
        preprocessor["upper"]
    )

    X_normalized = (
        X_clipped - preprocessor["mean"]
    ) / preprocessor["std"]

    return X_normalized.astype(np.float32)

In [ ]:
held_out_subject = 1

train_mask = subjects != held_out_subject
test_mask = subjects == held_out_subject

X_fold_train = X[train_mask]
X_fold_test = X[test_mask]

fold_preprocessor = fit_fold_preprocessor(
    X_fold_train
)

X_fold_train_processed = apply_fold_preprocessor(
    X_fold_train,
    fold_preprocessor
)

X_fold_test_processed = apply_fold_preprocessor(
    X_fold_test,
    fold_preprocessor
)

print("Training shape:", X_fold_train_processed.shape)
print("Held-out shape:", X_fold_test_processed.shape)

print(
    "Processed training mean:",
    X_fold_train_processed.mean(axis=(0, 1))
)

print(
    "Processed training std:",
    X_fold_train_processed.std(axis=(0, 1))
)

print(
    "Processed held-out mean:",
    X_fold_test_processed.mean(axis=(0, 1))
)

print(
    "Processed held-out std:",
    X_fold_test_processed.std(axis=(0, 1))
)

Training shape: (1834, 384, 30)
Held-out shape: (188, 384, 30)
Processed training mean: [-1.49773669e-07  3.91452943e-07  8.27834370e-08 -2.53716852e-08
  4.50687253e-07  3.21624952e-07 -1.05991046e-06  2.05872055e-07
  5.72332510e-07  3.79940843e-07 -2.25254979e-08  8.67815913e-07
 -5.87276929e-07  3.47693657e-07 -2.75803131e-07 -1.84290116e-07
  1.18141706e-07  8.13351392e-07 -1.22519373e-06 -3.41307555e-07
  5.73610862e-07  1.71589704e-07  2.71929594e-07  6.23611982e-07
 -2.09077751e-08 -7.91693139e-07 -4.01755145e-08  1.03849892e-07
  3.70217180e-07 -1.74800647e-07]
Processed training std: [0.9999617  1.0000169  0.9999606  1.0000621  0.9999483  0.9999995
 0.999962   0.99996376 1.0000051  0.9999535  0.9999646  0.9999714
 1.00006    1.0000603  0.9998848  1.000043   0.99989533 0.9999249
 0.99997824 0.9999936  0.9999284  0.99990857 0.99995035 0.99995875
 0.9999549  0.99998474 0.99994415 0.9999092  1.0000112  0.9999764 ]
Processed held-out mean: [ 0.00421397  0.00456419 -0.00628713  0.0

In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tensorflow.keras import layers, Model
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)
from sklearn.utils.class_weight import compute_class_weight

print("TensorFlow version:", tf.__version__)
print("NumPy loaded:", np.__name__)
print("X exists:", "X" in globals())
print("y exists:", "y" in globals())
print("subjects exists:", "subjects" in globals())

TensorFlow version: 2.21.0
NumPy loaded: numpy
X exists: True
y exists: True
subjects exists: True


In [ ]:
tf.keras.backend.clear_session()

def build_baseline_cnn(input_shape=(384, 30)):
    inputs = layers.Input(shape=input_shape)

    x = layers.Conv1D(
        filters=16,
        kernel_size=7,
        padding="same",
        use_bias=False
    )(inputs)

    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling1D(pool_size=4)(x)
    x = layers.Dropout(0.25)(x)

    x = layers.Conv1D(
        filters=32,
        kernel_size=5,
        padding="same",
        use_bias=False
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling1D(pool_size=4)(x)
    x = layers.Dropout(0.35)(x)

    x = layers.Conv1D(
        filters=64,
        kernel_size=3,
        padding="same",
        use_bias=False
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.GlobalAveragePooling1D()(x)

    x = layers.Dense(
        units=32,
        activation="relu"
    )(x)

    x = layers.Dropout(0.40)(x)

    outputs = layers.Dense(
        units=1,
        activation="sigmoid"
    )(x)

    return Model(
        inputs=inputs,
        outputs=outputs,
        name="Baseline_CNN"
    )

In [ ]:
held_out_subject = 1
inner_validation_subject = 2

train_subjects = [
    s for s in np.unique(subjects)
    if s not in [
        held_out_subject,
        inner_validation_subject
    ]
]

train_mask = np.isin(subjects, train_subjects)
validation_mask = subjects == inner_validation_subject
held_out_mask = subjects == held_out_subject

X_train_fold = X[train_mask]
y_train_fold = y[train_mask]

X_validation_fold = X[validation_mask]
y_validation_fold = y[validation_mask]

X_held_out_fold = X[held_out_mask]
y_held_out_fold = y[held_out_mask]

fold_preprocessor = fit_fold_preprocessor(
    X_train_fold
)

X_train_fold = apply_fold_preprocessor(
    X_train_fold,
    fold_preprocessor
)

X_validation_fold = apply_fold_preprocessor(
    X_validation_fold,
    fold_preprocessor
)

X_held_out_fold = apply_fold_preprocessor(
    X_held_out_fold,
    fold_preprocessor
)

print("Training:", X_train_fold.shape)
print("Validation:", X_validation_fold.shape)
print("Held-out:", X_held_out_fold.shape)

Training: (1702, 384, 30)
Validation: (132, 384, 30)
Held-out: (188, 384, 30)


In [ ]:
classes = np.unique(y_train_fold)

class_weight_values = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_fold
)

class_weights = {
    int(class_id): float(weight)
    for class_id, weight in zip(
        classes,
        class_weight_values
    )
}

print("Class weights:", class_weights)

Class weights: {0: 1.0, 1: 1.0}


In [ ]:
tf.keras.backend.clear_session()

smoke_model = build_baseline_cnn(
    input_shape=X_train_fold.shape[1:]
)

smoke_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4,
        clipnorm=1.0
    ),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.AUC(name="auc"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    )
]

history = smoke_model.fit(
    X_train_fold,
    y_train_fold,
    validation_data=(
        X_validation_fold,
        y_validation_fold
    ),
    epochs=40,
    batch_size=32,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/40
54/54 ━━━━━━━━━━━━━━━━━━━━ 4s 22ms/step - accuracy: 0.6022 - auc: 0.6310 - loss: 0.6779 - precision: 0.5781 - recall: 0.7568 - val_accuracy: 0.5000 - val_auc: 0.6439 - val_loss: 0.7082 - val_precision: 0.5000 - val_recall: 1.0000 - learning_rate: 1.0000e-04
Epoch 2/40
54/54 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.6569 - auc: 0.6956 - loss: 0.6524 - precision: 0.6446 - recall: 0.6992 - val_accuracy: 0.5000 - val_auc: 0.6591 - val_loss: 0.6993 - val_precision: 0.5000 - val_recall: 1.0000 - learning_rate: 1.0000e-04
Epoch 3/40
54/54 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.6722 - auc: 0.7160 - loss: 0.6325 - precision: 0.6619 - recall: 0.7039 - val_accuracy: 0.5076 - val_auc: 0.6626 - val_loss: 0.6881 - val_precision: 0.5039 - val_recall: 0.9697 - learning_rate: 1.0000e-04
Epoch 4/40
54/54 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.6704 - auc: 0.7252 - loss: 0.6211 - precision: 0.6659 - recall: 0.6839 - val_accuracy: 0.5682 - val_auc: 0.6625 - val_loss: 0.67

KeyboardInterrupt: 

In [ ]:
# Predict validation and held-out probabilities
validation_probability = smoke_model.predict(
    X_validation_fold,
    verbose=0
).ravel()

held_out_probability = smoke_model.predict(
    X_held_out_fold,
    verbose=0
).ravel()


# Select threshold using validation subject only
thresholds = np.arange(0.05, 0.96, 0.01)

best_threshold = 0.50
best_validation_score = -np.inf

for threshold in thresholds:
    validation_prediction = (
        validation_probability >= threshold
    ).astype(int)

    validation_score = balanced_accuracy_score(
        y_validation_fold,
        validation_prediction
    )

    if validation_score > best_validation_score:
        best_validation_score = validation_score
        best_threshold = threshold


# Evaluate held-out Subject 1
held_out_prediction = (
    held_out_probability >= best_threshold
).astype(int)

print("=" * 60)
print("SUBJECT 1 SMOKE TEST")
print("=" * 60)

print("Selected threshold:", best_threshold)
print(
    "Validation balanced accuracy:",
    best_validation_score
)

print(
    "Held-out accuracy:",
    accuracy_score(
        y_held_out_fold,
        held_out_prediction
    )
)

print(
    "Held-out balanced accuracy:",
    balanced_accuracy_score(
        y_held_out_fold,
        held_out_prediction
    )
)

print(
    "Held-out ROC-AUC:",
    roc_auc_score(
        y_held_out_fold,
        held_out_probability
    )
)

print("Held-out confusion matrix:")
print(
    confusion_matrix(
        y_held_out_fold,
        held_out_prediction
    )
)

print("Held-out classification report:")
print(
    classification_report(
        y_held_out_fold,
        held_out_prediction,
        target_names=["Alert", "Drowsy"],
        digits=4,
        zero_division=0
    )
)

SUBJECT 1 SMOKE TEST
Selected threshold: 0.6300000000000001
Validation balanced accuracy: 0.6742424242424242
Held-out accuracy: 0.7127659574468085
Held-out balanced accuracy: 0.7127659574468086
Held-out ROC-AUC: 0.8923721140787686
Held-out confusion matrix:
[[93  1]
 [53 41]]
Held-out classification report:
              precision    recall  f1-score   support

       Alert     0.6370    0.9894    0.7750        94
      Drowsy     0.9762    0.4362    0.6029        94

    accuracy                         0.7128       188
   macro avg     0.8066    0.7128    0.6890       188
weighted avg     0.8066    0.7128    0.6890       188



In [ ]:
# ============================================================
# DEFINE CNN + TRANSFORMER MODEL
# Run this cell before running the LOSO cell.
# ============================================================

def transformer_encoder_block(
    inputs,
    attention_heads=4,
    projection_dim=16,
    dropout_rate=0.20
):
    attention_output = layers.MultiHeadAttention(
        num_heads=attention_heads,
        key_dim=projection_dim,
        dropout=dropout_rate
    )(
        inputs,
        inputs
    )

    x = layers.Add()([
        inputs,
        attention_output
    ])

    x = layers.LayerNormalization(
        epsilon=1e-6
    )(x)

    feed_forward = layers.Dense(
        inputs.shape[-1] * 2,
        activation="gelu"
    )(x)

    feed_forward = layers.Dropout(
        dropout_rate
    )(feed_forward)

    feed_forward = layers.Dense(
        inputs.shape[-1]
    )(feed_forward)

    x = layers.Add()([
        x,
        feed_forward
    ])

    x = layers.LayerNormalization(
        epsilon=1e-6
    )(x)

    return x


def build_cnn_transformer(
    input_shape=(384, 30)
):
    inputs = layers.Input(
        shape=input_shape
    )

    # CNN feature extraction
    x = layers.Conv1D(
        filters=32,
        kernel_size=7,
        padding="same",
        use_bias=False
    )(inputs)

    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling1D(
        pool_size=2
    )(x)

    x = layers.Conv1D(
        filters=64,
        kernel_size=5,
        padding="same",
        use_bias=False
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling1D(
        pool_size=2
    )(x)

    # Transformer blocks
    x = transformer_encoder_block(
        x,
        attention_heads=4,
        projection_dim=16,
        dropout_rate=0.20
    )

    x = transformer_encoder_block(
        x,
        attention_heads=4,
        projection_dim=16,
        dropout_rate=0.20
    )

    # Classification head
    x = layers.GlobalAveragePooling1D()(x)

    x = layers.Dense(
        32,
        activation="relu"
    )(x)

    x = layers.Dropout(
        0.40
    )(x)

    outputs = layers.Dense(
        1,
        activation="sigmoid"
    )(x)

    return Model(
        inputs=inputs,
        outputs=outputs,
        name="CNN_Transformer"
    )


# Verify that the function exists and the model can be built
test_model = build_cnn_transformer(
    input_shape=X.shape[1:]
)

test_model.summary()

del test_model
tf.keras.backend.clear_session()

print("CNN + Transformer model is defined correctly.")

Model: "CNN_Transformer"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 384, 30)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 384, 32)   │      6,720 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 384, 32)   │        128 │ conv1d_3[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 384, 32)   │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_2     │ (None, 192, 32)   │          0 │ activation[0][0]  │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_4 (Conv1D)   │ (None, 192, 64)   │     10,240 │ max_pooling1d_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 192, 64)   │        256 │ conv1d_4[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 192, 64)   │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_3     │ (None, 96, 64)    │          0 │ activation_1[0][… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 96, 64)    │     16,640 │ max_pooling1d_3[… │
│ (MultiHeadAttentio… │                   │            │ max_pooling1d_3[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 96, 64)    │          0 │ max_pooling1d_3[… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 96, 64)    │        128 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 96, 128)   │      8,320 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 96, 128)   │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 96, 64)    │      8,256 │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 96, 64)    │          0 │ layer_normalizat… │
│                     │                   │            │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 96, 64)    │        128 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 96, 64)    │     16,640 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 86,401 (337.50 KB)

 Trainable params: 86,209 (336.75 KB)

 Non-trainable params: 192 (768.00 B)

CNN + Transformer model is defined correctly.


In [ ]:
# ============================================================
# COMPLETE 11-SUBJECT LOSO EVALUATION
# Add this as one new cell after the smoke-test evaluation cell.
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

subject_list = sorted(np.unique(subjects))

loso_results = []
loso_prediction_parts = []

for fold_number, held_out_subject in enumerate(subject_list):

    print("\n" + "=" * 70)
    print(
        f"FOLD {fold_number + 1}/{len(subject_list)} "
        f"- HELD-OUT SUBJECT: {held_out_subject}"
    )
    print("=" * 70)

    # --------------------------------------------------------
    # 1. Select an inner validation subject
    # --------------------------------------------------------
    remaining_subjects = [
        s for s in subject_list
        if s != held_out_subject
    ]

    # Deterministic rotating inner-validation subject
    inner_validation_subject = remaining_subjects[
        fold_number % len(remaining_subjects)
    ]

    inner_train_subjects = [
        s for s in remaining_subjects
        if s != inner_validation_subject
    ]

    print("Inner validation subject:", inner_validation_subject)
    print("Inner training subjects:", inner_train_subjects)

    # --------------------------------------------------------
    # 2. Create inner training and validation data
    # --------------------------------------------------------
    inner_train_mask = np.isin(
        subjects,
        inner_train_subjects
    )

    inner_validation_mask = (
        subjects == inner_validation_subject
    )

    outer_test_mask = (
        subjects == held_out_subject
    )

    X_inner_train = X[inner_train_mask]
    y_inner_train = y[inner_train_mask]

    X_inner_validation = X[inner_validation_mask]
    y_inner_validation = y[inner_validation_mask]

    X_outer_test = X[outer_test_mask]
    y_outer_test = y[outer_test_mask]

    # --------------------------------------------------------
    # 3. Fit preprocessing using inner training only
    # --------------------------------------------------------
    inner_preprocessor = fit_fold_preprocessor(
        X_inner_train
    )

    X_inner_train_processed = apply_fold_preprocessor(
        X_inner_train,
        inner_preprocessor
    )

    X_inner_validation_processed = apply_fold_preprocessor(
        X_inner_validation,
        inner_preprocessor
    )

    # --------------------------------------------------------
    # 4. Calculate class weights
    # --------------------------------------------------------
    inner_classes = np.unique(y_inner_train)

    inner_class_weight_values = compute_class_weight(
        class_weight="balanced",
        classes=inner_classes,
        y=y_inner_train
    )

    inner_class_weights = {
        int(class_id): float(weight)
        for class_id, weight in zip(
            inner_classes,
            inner_class_weight_values
        )
    }

    # --------------------------------------------------------
    # 5. Train inner model for threshold and epoch selection
    # --------------------------------------------------------
    tf.keras.backend.clear_session()

    inner_model = build_cnn_transformer(
        input_shape=X_inner_train_processed.shape[1:]
    )

    inner_model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=1e-4,
            clipnorm=1.0
        ),
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.AUC(name="auc")
        ]
    )

    inner_callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_auc",
            mode="max",
            patience=8,
            restore_best_weights=True,
            verbose=0
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            mode="min",
            factor=0.5,
            patience=3,
            min_lr=1e-7,
            verbose=0
        )
    ]

    inner_history = inner_model.fit(
        X_inner_train_processed,
        y_inner_train,
        validation_data=(
            X_inner_validation_processed,
            y_inner_validation
        ),
        epochs=40,
        batch_size=32,
        class_weight=inner_class_weights,
        callbacks=inner_callbacks,
        verbose=0
    )

    # --------------------------------------------------------
    # 6. Select threshold using inner validation only
    # --------------------------------------------------------
    inner_validation_probability = inner_model.predict(
        X_inner_validation_processed,
        verbose=0
    ).ravel()

    threshold_values = np.arange(
        0.05,
        0.96,
        0.01
    )

    selected_threshold = 0.50
    best_inner_balanced_accuracy = -np.inf

    for threshold in threshold_values:

        inner_validation_prediction = (
            inner_validation_probability >= threshold
        ).astype(int)

        inner_score = balanced_accuracy_score(
            y_inner_validation,
            inner_validation_prediction
        )

        if inner_score > best_inner_balanced_accuracy:
            best_inner_balanced_accuracy = inner_score
            selected_threshold = float(threshold)

    # Select training duration from the best inner validation epoch
    validation_auc_history = np.asarray(
        inner_history.history["val_auc"]
    )

    best_epoch = int(
        np.argmax(validation_auc_history) + 1
    )

    print(
        "Selected threshold:",
        round(selected_threshold, 3)
    )

    print(
        "Inner validation balanced accuracy:",
        round(best_inner_balanced_accuracy, 4)
    )

    print("Selected training epochs:", best_epoch)

    # --------------------------------------------------------
    # 7. Fit preprocessing using all non-held-out subjects
    # --------------------------------------------------------
    outer_train_mask = (
        subjects != held_out_subject
    )

    X_outer_train = X[outer_train_mask]
    y_outer_train = y[outer_train_mask]

    outer_preprocessor = fit_fold_preprocessor(
        X_outer_train
    )

    X_outer_train_processed = apply_fold_preprocessor(
        X_outer_train,
        outer_preprocessor
    )

    X_outer_test_processed = apply_fold_preprocessor(
        X_outer_test,
        outer_preprocessor
    )

    # --------------------------------------------------------
    # 8. Train final outer-fold model
    # --------------------------------------------------------
    outer_classes = np.unique(y_outer_train)

    outer_class_weight_values = compute_class_weight(
        class_weight="balanced",
        classes=outer_classes,
        y=y_outer_train
    )

    outer_class_weights = {
        int(class_id): float(weight)
        for class_id, weight in zip(
            outer_classes,
            outer_class_weight_values
        )
    }

    tf.keras.backend.clear_session()

    outer_model = build_cnn_transformer(
        input_shape=X_outer_train_processed.shape[1:]
    )

    outer_model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=1e-4,
            clipnorm=1.0
        ),
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.AUC(name="auc")
        ]
    )

    outer_model.fit(
        X_outer_train_processed,
        y_outer_train,
        epochs=best_epoch,
        batch_size=32,
        class_weight=outer_class_weights,
        verbose=0
    )

    # --------------------------------------------------------
    # 9. Evaluate held-out subject
    # --------------------------------------------------------
    outer_probability = outer_model.predict(
        X_outer_test_processed,
        verbose=0
    ).ravel()

    outer_prediction = (
        outer_probability >= selected_threshold
    ).astype(int)

    fold_accuracy = accuracy_score(
        y_outer_test,
        outer_prediction
    )

    fold_balanced_accuracy = balanced_accuracy_score(
        y_outer_test,
        outer_prediction
    )

    fold_auc = roc_auc_score(
        y_outer_test,
        outer_probability
    )

    fold_precision = precision_score(
        y_outer_test,
        outer_prediction,
        zero_division=0
    )

    fold_recall = recall_score(
        y_outer_test,
        outer_prediction,
        zero_division=0
    )

    fold_f1 = f1_score(
        y_outer_test,
        outer_prediction,
        zero_division=0
    )

    fold_result = {
        "held_out_subject": int(held_out_subject),
        "inner_validation_subject": int(
            inner_validation_subject
        ),
        "selected_threshold": selected_threshold,
        "best_epoch": best_epoch,
        "samples": len(y_outer_test),
        "accuracy": fold_accuracy,
        "balanced_accuracy": fold_balanced_accuracy,
        "auc": fold_auc,
        "precision": fold_precision,
        "recall": fold_recall,
        "f1": fold_f1
    }

    loso_results.append(fold_result)

    fold_predictions = pd.DataFrame({
        "subject_id": subjects[outer_test_mask],
        "true_label": y_outer_test,
        "probability": outer_probability,
        "prediction": outer_prediction,
        "threshold": selected_threshold
    })

    loso_prediction_parts.append(
        fold_predictions
    )

    print("Samples:", len(y_outer_test))
    print("Accuracy:", round(fold_accuracy, 4))
    print(
        "Balanced accuracy:",
        round(fold_balanced_accuracy, 4)
    )
    print("ROC-AUC:", round(fold_auc, 4))
    print("F1:", round(fold_f1, 4))


# ============================================================
# 10. Subject-level results
# ============================================================

loso_results_df = pd.DataFrame(loso_results)

print("\n" + "=" * 70)
print("LOSO RESULTS BY SUBJECT")
print("=" * 70)

display(loso_results_df)


# ============================================================
# 11. Pooled predictions and metrics
# ============================================================

loso_predictions_df = pd.concat(
    loso_prediction_parts,
    ignore_index=True
)

pooled_y_true = (
    loso_predictions_df["true_label"].to_numpy()
)

pooled_probability = (
    loso_predictions_df["probability"].to_numpy()
)

pooled_prediction = (
    loso_predictions_df["prediction"].to_numpy()
)

pooled_accuracy = accuracy_score(
    pooled_y_true,
    pooled_prediction
)

pooled_balanced_accuracy = balanced_accuracy_score(
    pooled_y_true,
    pooled_prediction
)

pooled_auc = roc_auc_score(
    pooled_y_true,
    pooled_probability
)

print("\n" + "=" * 70)
print("POOLED LOSO METRICS")
print("=" * 70)

print("Pooled accuracy:", pooled_accuracy)
print(
    "Pooled balanced accuracy:",
    pooled_balanced_accuracy
)
print("Pooled ROC-AUC:", pooled_auc)

print("\nPooled confusion matrix:")
print(
    confusion_matrix(
        pooled_y_true,
        pooled_prediction
    )
)


# ============================================================
# 12. Mean and standard deviation
# ============================================================

metric_names = [
    "accuracy",
    "balanced_accuracy",
    "auc",
    "precision",
    "recall",
    "f1"
]

summary_rows = []

for metric_name in metric_names:

    values = loso_results_df[
        metric_name
    ].to_numpy()

    summary_rows.append({
        "metric": metric_name,
        "mean": np.mean(values),
        "std": np.std(values, ddof=1)
    })

loso_summary_df = pd.DataFrame(
    summary_rows
)

print("\n" + "=" * 70)
print("LOSO MEAN ± STANDARD DEVIATION")
print("=" * 70)

display(loso_summary_df)


# ============================================================
# 13. Save results
# ============================================================

loso_results_df.to_csv(
    "/content/cnn_transformer_loso_by_subject.csv",
    index=False
)

loso_summary_df.to_csv(
    "/content/cnn_transformer_loso_mean_std.csv",
    index=False
)

loso_predictions_df.to_csv(
    "/content/cnn_transformer_loso_predictions.csv",
    index=False
)

print("CNN + Transformer results saved.")


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model


def build_multiscale_cnn(input_shape=(384, 30)):

    inputs = layers.Input(
        shape=input_shape,
        name="input_layer"
    )

    branch_outputs = []

    # Three parallel convolution branches
    for kernel_size in [3, 5, 7]:

        branch = layers.Conv1D(
            filters=32,
            kernel_size=kernel_size,
            padding="same",
            use_bias=False,
            name=f"conv_kernel_{kernel_size}"
        )(inputs)

        branch = layers.BatchNormalization(
            name=f"batch_norm_{kernel_size}"
        )(branch)

        branch = layers.Activation(
            "relu",
            name=f"relu_{kernel_size}"
        )(branch)

        branch = layers.MaxPooling1D(
            pool_size=2,
            name=f"pool_{kernel_size}"
        )(branch)

        branch_outputs.append(branch)

    # Combine the three branches
    x = layers.Concatenate(
        name="multiscale_concatenate"
    )(branch_outputs)

    # Feature fusion
    x = layers.Conv1D(
        filters=64,
        kernel_size=3,
        padding="same",
        use_bias=False,
        name="fusion_conv"
    )(x)

    x = layers.BatchNormalization(
        name="fusion_batch_norm"
    )(x)

    x = layers.Activation(
        "relu",
        name="fusion_relu"
    )(x)

    # Classification head
    x = layers.GlobalAveragePooling1D(
        name="global_average_pooling"
    )(x)

    x = layers.Dense(
        32,
        activation="relu",
        name="dense_32"
    )(x)

    x = layers.Dropout(
        0.35,
        name="dropout"
    )(x)

    outputs = layers.Dense(
        1,
        activation="sigmoid",
        name="drowsiness_output"
    )(x)

    return Model(
        inputs=inputs,
        outputs=outputs,
        name="multiscale_1d_cnn"
    )


# Build test model
test_multiscale_model = build_multiscale_cnn(
    input_shape=(384, 30)
)

# Display architecture
test_multiscale_model.summary()

print(
    "Multi-scale CNN parameters:",
    test_multiscale_model.count_params()
)

Model: "multiscale_1d_cnn"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 384, 30)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_kernel_3       │ (None, 384, 32)   │      2,880 │ input_layer[0][0] │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_kernel_5       │ (None, 384, 32)   │      4,800 │ input_layer[0][0] │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_kernel_7       │ (None, 384, 32)   │      6,720 │ input_layer[0][0] │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_norm_3        │ (None, 384, 32)   │        128 │ conv_kernel_3[0]… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_norm_5        │ (None, 384, 32)   │        128 │ conv_kernel_5[0]… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_norm_7        │ (None, 384, 32)   │        128 │ conv_kernel_7[0]… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ relu_3 (Activation) │ (None, 384, 32)   │          0 │ batch_norm_3[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ relu_5 (Activation) │ (None, 384, 32)   │          0 │ batch_norm_5[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ relu_7 (Activation) │ (None, 384, 32)   │          0 │ batch_norm_7[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool_3              │ (None, 192, 32)   │          0 │ relu_3[0][0]      │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool_5              │ (None, 192, 32)   │          0 │ relu_5[0][0]      │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool_7              │ (None, 192, 32)   │          0 │ relu_7[0][0]      │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiscale_concate… │ (None, 192, 96)   │          0 │ pool_3[0][0],     │
│ (Concatenate)       │                   │            │ pool_5[0][0],     │
│                     │                   │            │ pool_7[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fusion_conv         │ (None, 192, 64)   │     18,432 │ multiscale_conca… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fusion_batch_norm   │ (None, 192, 64)   │        256 │ fusion_conv[0][0] │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fusion_relu         │ (None, 192, 64)   │          0 │ fusion_batch_nor… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 35,585 (139.00 KB)

 Trainable params: 35,265 (137.75 KB)

 Non-trainable params: 320 (1.25 KB)

Multi-scale CNN parameters: 35585


In [ ]:
# ============================================================
# COMPLETE 11-SUBJECT LOSO EVALUATION
# Add this as one new cell after the smoke-test evaluation cell.
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

subject_list = sorted(np.unique(subjects))

loso_results = []
loso_prediction_parts = []

for fold_number, held_out_subject in enumerate(subject_list):

    print("\n" + "=" * 70)
    print(
        f"FOLD {fold_number + 1}/{len(subject_list)} "
        f"- HELD-OUT SUBJECT: {held_out_subject}"
    )
    print("=" * 70)

    # --------------------------------------------------------
    # 1. Select an inner validation subject
    # --------------------------------------------------------
    remaining_subjects = [
        s for s in subject_list
        if s != held_out_subject
    ]

    # Deterministic rotating inner-validation subject
    inner_validation_subject = remaining_subjects[
        fold_number % len(remaining_subjects)
    ]

    inner_train_subjects = [
        s for s in remaining_subjects
        if s != inner_validation_subject
    ]

    print("Inner validation subject:", inner_validation_subject)
    print("Inner training subjects:", inner_train_subjects)

    # --------------------------------------------------------
    # 2. Create inner training and validation data
    # --------------------------------------------------------
    inner_train_mask = np.isin(
        subjects,
        inner_train_subjects
    )

    inner_validation_mask = (
        subjects == inner_validation_subject
    )

    outer_test_mask = (
        subjects == held_out_subject
    )

    X_inner_train = X[inner_train_mask]
    y_inner_train = y[inner_train_mask]

    X_inner_validation = X[inner_validation_mask]
    y_inner_validation = y[inner_validation_mask]

    X_outer_test = X[outer_test_mask]
    y_outer_test = y[outer_test_mask]

    # --------------------------------------------------------
    # 3. Fit preprocessing using inner training only
    # --------------------------------------------------------
    inner_preprocessor = fit_fold_preprocessor(
        X_inner_train
    )

    X_inner_train_processed = apply_fold_preprocessor(
        X_inner_train,
        inner_preprocessor
    )

    X_inner_validation_processed = apply_fold_preprocessor(
        X_inner_validation,
        inner_preprocessor
    )

    # --------------------------------------------------------
    # 4. Calculate class weights
    # --------------------------------------------------------
    inner_classes = np.unique(y_inner_train)

    inner_class_weight_values = compute_class_weight(
        class_weight="balanced",
        classes=inner_classes,
        y=y_inner_train
    )

    inner_class_weights = {
        int(class_id): float(weight)
        for class_id, weight in zip(
            inner_classes,
            inner_class_weight_values
        )
    }

    # --------------------------------------------------------
    # 5. Train inner model for threshold and epoch selection
    # --------------------------------------------------------
    tf.keras.backend.clear_session()

    inner_model = build_multiscale_cnn(
        input_shape=X_inner_train_processed.shape[1:]
    )

    inner_model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=1e-4,
            clipnorm=1.0
        ),
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.AUC(name="auc")
        ]
    )
    inner_callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_auc",
            mode="max",
            patience=8,
            restore_best_weights=True,
            verbose=0
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            mode="min",
            factor=0.5,
            patience=3,
            min_lr=1e-7,
            verbose=0
        )
    ]

    inner_history = inner_model.fit(
        X_inner_train_processed,
        y_inner_train,
        validation_data=(
            X_inner_validation_processed,
            y_inner_validation
        ),
        epochs=40,
        batch_size=32,
        class_weight=inner_class_weights,
        callbacks=inner_callbacks,
        verbose=0
    )

    # --------------------------------------------------------
    # 6. Select threshold using inner validation only
    # --------------------------------------------------------
    inner_validation_probability = inner_model.predict(
        X_inner_validation_processed,
        verbose=0
    ).ravel()

    threshold_values = np.arange(
        0.05,
        0.96,
        0.01
    )

    selected_threshold = 0.50
    best_inner_balanced_accuracy = -np.inf

    for threshold in threshold_values:

        inner_validation_prediction = (
            inner_validation_probability >= threshold
        ).astype(int)

        inner_score = balanced_accuracy_score(
            y_inner_validation,
            inner_validation_prediction
        )

        if inner_score > best_inner_balanced_accuracy:
            best_inner_balanced_accuracy = inner_score
            selected_threshold = float(threshold)

    # Select training duration from the best inner validation epoch
    validation_auc_history = np.asarray(
        inner_history.history["val_auc"]
    )

    best_epoch = int(
        np.argmax(validation_auc_history) + 1
    )

    print(
        "Selected threshold:",
        round(selected_threshold, 3)
    )

    print(
        "Inner validation balanced accuracy:",
        round(best_inner_balanced_accuracy, 4)
    )

    print("Selected training epochs:", best_epoch)

    # --------------------------------------------------------
    # 7. Fit preprocessing using all non-held-out subjects
    # --------------------------------------------------------
    outer_train_mask = (
        subjects != held_out_subject
    )

    X_outer_train = X[outer_train_mask]
    y_outer_train = y[outer_train_mask]

    outer_preprocessor = fit_fold_preprocessor(
        X_outer_train
    )

    X_outer_train_processed = apply_fold_preprocessor(
        X_outer_train,
        outer_preprocessor
    )

    X_outer_test_processed = apply_fold_preprocessor(
        X_outer_test,
        outer_preprocessor
    )

    # --------------------------------------------------------
    # 8. Train final outer-fold model
    # --------------------------------------------------------
    outer_classes = np.unique(y_outer_train)

    outer_class_weight_values = compute_class_weight(
        class_weight="balanced",
        classes=outer_classes,
        y=y_outer_train
    )

    outer_class_weights = {
        int(class_id): float(weight)
        for class_id, weight in zip(
            outer_classes,
            outer_class_weight_values
        )
    }

    tf.keras.backend.clear_session()

    outer_model = build_multiscale_cnn(
    input_shape=X_outer_train_processed.shape[1:]
)

    outer_model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=1e-4,
            clipnorm=1.0
        ),
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.AUC(name="auc")
        ]
    )

    outer_model.fit(
        X_outer_train_processed,
        y_outer_train,
        epochs=best_epoch,
        batch_size=32,
        class_weight=outer_class_weights,
        verbose=0
    )

    # --------------------------------------------------------
    # 9. Evaluate held-out subject
    # --------------------------------------------------------
    outer_probability = outer_model.predict(
        X_outer_test_processed,
        verbose=0
    ).ravel()

    outer_prediction = (
        outer_probability >= selected_threshold
    ).astype(int)

    fold_accuracy = accuracy_score(
        y_outer_test,
        outer_prediction
    )

    fold_balanced_accuracy = balanced_accuracy_score(
        y_outer_test,
        outer_prediction
    )

    fold_auc = roc_auc_score(
        y_outer_test,
        outer_probability
    )

    fold_precision = precision_score(
        y_outer_test,
        outer_prediction,
        zero_division=0
    )

    fold_recall = recall_score(
        y_outer_test,
        outer_prediction,
        zero_division=0
    )

    fold_f1 = f1_score(
        y_outer_test,
        outer_prediction,
        zero_division=0
    )

    fold_result = {
        "held_out_subject": int(held_out_subject),
        "inner_validation_subject": int(
            inner_validation_subject
        ),
        "selected_threshold": selected_threshold,
        "best_epoch": best_epoch,
        "samples": len(y_outer_test),
        "accuracy": fold_accuracy,
        "balanced_accuracy": fold_balanced_accuracy,
        "auc": fold_auc,
        "precision": fold_precision,
        "recall": fold_recall,
        "f1": fold_f1
    }

    loso_results.append(fold_result)

    fold_predictions = pd.DataFrame({
        "subject_id": subjects[outer_test_mask],
        "true_label": y_outer_test,
        "probability": outer_probability,
        "prediction": outer_prediction,
        "threshold": selected_threshold
    })

    loso_prediction_parts.append(
        fold_predictions
    )

    print("Samples:", len(y_outer_test))
    print("Accuracy:", round(fold_accuracy, 4))
    print(
        "Balanced accuracy:",
        round(fold_balanced_accuracy, 4)
    )
    print("ROC-AUC:", round(fold_auc, 4))
    print("F1:", round(fold_f1, 4))


# ============================================================
# 10. Subject-level results
# ============================================================

loso_results_df = pd.DataFrame(loso_results)

print("\n" + "=" * 70)
print("LOSO RESULTS BY SUBJECT")
print("=" * 70)

display(loso_results_df)


# ============================================================
# 11. Pooled predictions and metrics
# ============================================================

loso_predictions_df = pd.concat(
    loso_prediction_parts,
    ignore_index=True
)

pooled_y_true = (
    loso_predictions_df["true_label"].to_numpy()
)

pooled_probability = (
    loso_predictions_df["probability"].to_numpy()
)

pooled_prediction = (
    loso_predictions_df["prediction"].to_numpy()
)

pooled_accuracy = accuracy_score(
    pooled_y_true,
    pooled_prediction
)

pooled_balanced_accuracy = balanced_accuracy_score(
    pooled_y_true,
    pooled_prediction
)

pooled_auc = roc_auc_score(
    pooled_y_true,
    pooled_probability
)

print("\n" + "=" * 70)
print("POOLED LOSO METRICS")
print("=" * 70)

print("Pooled accuracy:", pooled_accuracy)
print(
    "Pooled balanced accuracy:",
    pooled_balanced_accuracy
)
print("Pooled ROC-AUC:", pooled_auc)

print("\nPooled confusion matrix:")
print(
    confusion_matrix(
        pooled_y_true,
        pooled_prediction
    )
)


# ============================================================
# 12. Mean and standard deviation
# ============================================================

metric_names = [
    "accuracy",
    "balanced_accuracy",
    "auc",
    "precision",
    "recall",
    "f1"
]

summary_rows = []

for metric_name in metric_names:

    values = loso_results_df[
        metric_name
    ].to_numpy()

    summary_rows.append({
        "metric": metric_name,
        "mean": np.mean(values),
        "std": np.std(values, ddof=1)
    })

loso_summary_df = pd.DataFrame(
    summary_rows
)

print("\n" + "=" * 70)
print("LOSO MEAN ± STANDARD DEVIATION")
print("=" * 70)

display(loso_summary_df)


# ============================================================
# 13. Save results
# ============================================================

loso_results_df.to_csv(
    "/content/multiscale_cnn_loso_by_subject.csv",
    index=False
)

loso_summary_df.to_csv(
    "/content/multiscale_cnn_loso_mean_std.csv",
    index=False
)

loso_predictions_df.to_csv(
    "/content/multiscale_cnn_loso_predictions.csv",
    index=False
)

print("Multi-scale CNN results saved.")


FOLD 1/11 - HELD-OUT SUBJECT: 1
Inner validation subject: 2
Inner training subjects: [np.int32(3), np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10), np.int32(11)]
Selected threshold: 0.74
Inner validation balanced accuracy: 0.7045
Selected training epochs: 1
Samples: 188
Accuracy: 0.6223
Balanced accuracy: 0.6223
ROC-AUC: 0.8796
F1: 0.3932

FOLD 2/11 - HELD-OUT SUBJECT: 2
Inner validation subject: 3
Inner training subjects: [np.int32(1), np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10), np.int32(11)]


Selected threshold: 0.41
Inner validation balanced accuracy: 0.8133
Selected training epochs: 29


Samples: 132
Accuracy: 0.5379
Balanced accuracy: 0.5379
ROC-AUC: 0.6267
F1: 0.6115

FOLD 3/11 - HELD-OUT SUBJECT: 3
Inner validation subject: 4
Inner training subjects: [np.int32(1), np.int32(2), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10), np.int32(11)]
Selected threshold: 0.58
Inner validation balanced accuracy: 0.7905
Selected training epochs: 8
Samples: 150
Accuracy: 0.68
Balanced accuracy: 0.68
ROC-AUC: 0.8112
F1: 0.5472

FOLD 4/11 - HELD-OUT SUBJECT: 4
Inner validation subject: 5
Inner training subjects: [np.int32(1), np.int32(2), np.int32(3), np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10), np.int32(11)]
Selected threshold: 0.5
Inner validation balanced accuracy: 0.875
Selected training epochs: 4
Samples: 148
Accuracy: 0.5068
Balanced accuracy: 0.5068
ROC-AUC: 0.8009
F1: 0.6697

FOLD 5/11 - HELD-OUT SUBJECT: 5
Inner validation subject: 6
Inner training subjects: [np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(7), n

,held_out_subject,inner_validation_subject,selected_threshold,best_epoch,samples,accuracy,balanced_accuracy,auc,precision,recall,f1
0,1,2,0.74,1,188,0.622340,0.622340,0.879584,1.000000,0.244681,0.393162
1,2,3,0.41,29,132,0.537879,0.537879,0.626722,0.527473,0.727273,0.611465
2,3,4,0.58,8,150,0.680000,0.680000,0.811200,0.935484,0.386667,0.547170
3,4,5,0.50,4,148,0.506757,0.506757,0.800950,0.503401,1.000000,0.669683
4,5,6,0.30,27,224,0.892857,0.892857,0.967554,0.860656,0.937500,0.897436
5,6,7,0.59,7,166,0.825301,0.825301,0.883583,0.950000,0.686747,0.797203
6,7,8,0.75,22,102,0.627451,0.627451,0.628989,0.675676,0.490196,0.568182
7,8,9,0.41,11,264,0.689394,0.689394,0.801251,0.637363,0.878788,0.738854
8,9,10,0.65,2,314,0.815287,0.815287,0.913749,0.909091,0.700637,0.791367
9,10,11,0.76,4,108,0.750000,0.750000,0.881344,1.000000,0.500000,0.666667



POOLED LOSO METRICS
Pooled accuracy: 0.7037586547972304
Pooled balanced accuracy: 0.7037586547972304
Pooled ROC-AUC: 0.7707306669171263

Pooled confusion matrix:
[[708 303]
 [296 715]]

LOSO MEAN ± STANDARD DEVIATION


,metric,mean,std
0,accuracy,0.687080,0.122585
1,balanced_accuracy,0.687080,0.122585
2,auc,0.813056,0.110048
3,precision,0.779069,0.197074
4,recall,0.676936,0.245053
5,f1,0.670704,0.139143


Multi-scale CNN results saved.


In [ ]:
# ============================================================
# CNN + TRANSFORMER MODEL
# ============================================================

def transformer_encoder_block(
    inputs,
    attention_heads=4,
    projection_dim=32,
    dropout_rate=0.20
):
    attention_output = layers.MultiHeadAttention(
        num_heads=attention_heads,
        key_dim=projection_dim,
        dropout=dropout_rate
    )(
        inputs,
        inputs
    )

    x = layers.Add()([
        inputs,
        attention_output
    ])

    x = layers.LayerNormalization(
        epsilon=1e-6
    )(x)

    feed_forward = layers.Dense(
        projection_dim * 2,
        activation="gelu"
    )(x)

    feed_forward = layers.Dropout(
        dropout_rate
    )(feed_forward)

    feed_forward = layers.Dense(
        inputs.shape[-1]
    )(feed_forward)

    x = layers.Add()([
        x,
        feed_forward
    ])

    x = layers.LayerNormalization(
        epsilon=1e-6
    )(x)

    return x


def build_cnn_transformer(
    input_shape=(384, 30)
):
    inputs = layers.Input(
        shape=input_shape
    )

    # Local temporal feature extraction
    x = layers.Conv1D(
        filters=32,
        kernel_size=7,
        padding="same",
        use_bias=False
    )(inputs)

    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling1D(
        pool_size=2
    )(x)

    x = layers.Conv1D(
        filters=64,
        kernel_size=5,
        padding="same",
        use_bias=False
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling1D(
        pool_size=2
    )(x)

    # Transformer attention blocks
    x = transformer_encoder_block(
        x,
        attention_heads=4,
        projection_dim=16,
        dropout_rate=0.20
    )

    x = transformer_encoder_block(
        x,
        attention_heads=4,
        projection_dim=16,
        dropout_rate=0.20
    )

    # Global classification head
    x = layers.GlobalAveragePooling1D()(x)

    x = layers.Dense(
        32,
        activation="relu"
    )(x)

    x = layers.Dropout(
        0.40
    )(x)

    outputs = layers.Dense(
        1,
        activation="sigmoid"
    )(x)

    return Model(
        inputs=inputs,
        outputs=outputs,
        name="CNN_Transformer"
    )


test_model = build_cnn_transformer(
    input_shape=X.shape[1:]
)

test_model.summary()

del test_model
tf.keras.backend.clear_session()

NameError: name 'layers' is not defined

In [ ]:
comparison = pd.DataFrame({
    "model": [
        "1D CNN",
        "CNN + Transformer"
    ],
    "mean_balanced_accuracy": [
        0.680627,
        0.657499
    ],
    "std_balanced_accuracy": [
        0.114435,
        0.112790
    ],
    "mean_auc": [
        0.804944,
        0.798597
    ],
    "std_auc": [
        0.110488,
        0.109040
    ]
})

display(comparison)

,model,mean_balanced_accuracy,std_balanced_accuracy,mean_auc,std_auc
0,1D CNN,0.680627,0.114435,0.804944,0.110488
1,CNN + Transformer,0.657499,0.112790,0.798597,0.109040


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os

SAVE_DIR = "/content/drive/MyDrive/eeg_models"
os.makedirs(SAVE_DIR, exist_ok=True)

print("Saving to:", SAVE_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saving to: /content/drive/MyDrive/eeg_models


In [ ]:
# Fit preprocessing on all available data
# This model is for future prediction after LOSO evaluation is complete.

final_preprocessor = fit_fold_preprocessor(X)

X_all_processed = apply_fold_preprocessor(
    X,
    final_preprocessor
)

print("Processed data shape:", X_all_processed.shape)
print("Processed mean:", X_all_processed.mean())
print("Processed std:", X_all_processed.std())

Processed data shape: (2022, 384, 30)
Processed mean: 4.058147e-10
Processed std: 0.9999996


In [ ]:
# Create class weights for final training

classes = np.unique(y)

class_weight_values = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y
)

final_class_weights = {
    int(class_id): float(weight)
    for class_id, weight in zip(
        classes,
        class_weight_values
    )
}

print("Final class weights:", final_class_weights)

Final class weights: {0: 1.0, 1: 1.0}


In [ ]:
tf.keras.backend.clear_session()

final_transformer_model = build_cnn_transformer(
    input_shape=X_all_processed.shape[1:]
)

final_transformer_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4,
        clipnorm=1.0
    ),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.AUC(name="auc"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)

final_transformer_model.fit(
    X_all_processed,
    y,
    epochs=final_transformer_epochs,
    batch_size=32,
    class_weight=final_class_weights,
    verbose=1
)

transformer_model_path = os.path.join(
    SAVE_DIR,
    "final_cnn_transformer.keras"
)

final_transformer_model.save(transformer_model_path)

print("Saved CNN + Transformer:")
print(transformer_model_path)

In [ ]:
np.savez(
    os.path.join(
        SAVE_DIR,
        "final_preprocessor.npz"
    ),
    lower=final_preprocessor["lower"],
    upper=final_preprocessor["upper"],
    mean=final_preprocessor["mean"],
    std=final_preprocessor["std"]
)

print("Saved preprocessing parameters.")

In [ ]:
import shutil

result_files = [
    "baseline_cnn_loso_by_subject.csv",
    "baseline_cnn_loso_mean_std.csv",
    "baseline_cnn_loso_predictions.csv",
    "cnn_transformer_loso_by_subject.csv",
    "cnn_transformer_loso_mean_std.csv",
    "cnn_transformer_loso_predictions.csv"
]

for filename in result_files:
    source_path = os.path.join(
        "/content",
        filename
    )

    destination_path = os.path.join(
        SAVE_DIR,
        filename
    )

    if os.path.exists(source_path):
        shutil.copy2(
            source_path,
            destination_path
        )
        print("Copied:", filename)
    else:
        print("Not found:", source_path)

In [ ]:
print("Files saved in Google Drive:\n")

for filename in sorted(
    os.listdir(SAVE_DIR)
):
    file_path = os.path.join(
        SAVE_DIR,
        filename
    )

    file_size_mb = (
        os.path.getsize(file_path)
        / (1024 * 1024)
    )

    print(
        f"{filename}: "
        f"{file_size_mb:.2f} MB"
    )

In [ ]:
import shutil

result_files = [
    "baseline_cnn_loso_by_subject.csv",
    "baseline_cnn_loso_mean_std.csv",
    "baseline_cnn_loso_predictions.csv",
    "cnn_transformer_loso_by_subject.csv",
    "cnn_transformer_loso_mean_std.csv",
    "cnn_transformer_loso_predictions.csv"
]

for filename in result_files:
    source_path = os.path.join(
        "/content",
        filename
    )

    destination_path = os.path.join(
        SAVE_DIR,
        filename
    )

    if os.path.exists(source_path):
        shutil.copy2(
            source_path,
            destination_path
        )
        print("Copied:", filename)
    else:
        print("Not found:", filename)

In [ ]:
import os

required_files = [
    "final_cnn_transformer.keras",
    "final_preprocessor.npz",
    "classification_thresholds.json"
]

for filename in required_files:
    path = os.path.join(SAVE_DIR, filename)
    print(filename, "->", os.path.exists(path))

NameError: name 'SAVE_DIR' is not defined

In [ ]:
# ============================================================
# TRAIN AND SAVE FINAL 1D CNN
# ============================================================

import os
import numpy as np
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight

# Make sure the Drive folder exists
SAVE_DIR = "/content/drive/MyDrive/eeg_models"
os.makedirs(SAVE_DIR, exist_ok=True)

# Preprocess all data using the existing preprocessing functions
final_preprocessor = fit_fold_preprocessor(X)

X_all_processed = apply_fold_preprocessor(
    X,
    final_preprocessor
)

# Class weights
classes = np.unique(y)

class_weight_values = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y
)

final_class_weights = {
    int(class_id): float(weight)
    for class_id, weight in zip(
        classes,
        class_weight_values
    )
}

# Use the median epoch count from the completed 1D CNN LOSO results
cnn_results = pd.read_csv(
    "/content/baseline_cnn_loso_by_subject.csv"
)

final_cnn_epochs = int(
    round(
        cnn_results["best_epoch"].median()
    )
)

print("Final CNN epochs:", final_cnn_epochs)
print("Class weights:", final_class_weights)
print("Processed data shape:", X_all_processed.shape)

# Build and train final 1D CNN
tf.keras.backend.clear_session()

final_cnn_model = build_baseline_cnn(
    input_shape=X_all_processed.shape[1:]
)

final_cnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4,
        clipnorm=1.0
    ),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.AUC(name="auc"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)

final_cnn_model.fit(
    X_all_processed,
    y,
    epochs=final_cnn_epochs,
    batch_size=32,
    class_weight=final_class_weights,
    verbose=1
)

# Save the model
cnn_model_path = os.path.join(
    SAVE_DIR,
    "final_1d_cnn.keras"
)

final_cnn_model.save(cnn_model_path)

print("Final 1D CNN saved successfully:")
print(cnn_model_path)

In [ ]:
import numpy as np

test_sample = X[0:1].astype("float32")

print("Test sample shape:", test_sample.shape)

probability = float(
    model.predict(
        test_sample,
        verbose=0
    )[0][0]
)

print("Probability:", probability)

if probability >= 0.5:
    print("Prediction: Drowsy")
else:
    print("Prediction: Alert")

print("True label:", y[0])

In [ ]:
import numpy as np

PREPROCESSOR_PATH = (
    "/content/drive/MyDrive/"
    "EEG_Drowsiness_New/Models/"
    "final_preprocessor.npz"
)

preprocessor = np.load(PREPROCESSOR_PATH)

lower = preprocessor["lower"].astype("float32")
upper = preprocessor["upper"].astype("float32")
mean = preprocessor["mean"].astype("float32")
std = preprocessor["std"].astype("float32")

# Keep the original raw data
X_raw = X.copy()

# Apply the same preprocessing used during training
X_processed = np.clip(
    X_raw,
    lower,
    upper
)

X_processed = (
    X_processed - mean
) / (
    std + 1e-8
)

X_processed = X_processed.astype("float32")

print("Processed shape:", X_processed.shape)
print("Processed minimum:", X_processed.min())
print("Processed maximum:", X_processed.max())
print("Processed mean:", X_processed.mean())
print("Processed standard deviation:", X_processed.std())

In [ ]:
probabilities_processed = model.predict(
    X_processed,
    verbose=0
).reshape(-1)

predictions_processed = (
    probabilities_processed >= 0.5
).astype("int32")

accuracy_processed = np.mean(
    predictions_processed == y
)

print("Corrected accuracy:", round(float(accuracy_processed), 4))
print(
    "Correct predictions:",
    int(np.sum(predictions_processed == y))
)
print("Total samples:", len(y))

NameError: name 'model' is not defined

In [ ]:
import json
import os

MODEL_DIR = (
    "/content/drive/MyDrive/"
    "EEG_Drowsiness_New/Models"
)

threshold_path = os.path.join(
    MODEL_DIR,
    "classification_thresholds.json"
)

with open(threshold_path, "w") as file:
    json.dump(
        {"cnn_threshold": 0.50},
        file,
        indent=4
    )

print("Saved threshold:", threshold_path)

Saved threshold: /content/drive/MyDrive/EEG_Drowsiness_New/Models/classification_thresholds.json


In [ ]:
import tensorflow as tf
import os

MODEL_PATH = os.path.join(
    MODEL_DIR,
    "final_1d_cnn.keras"
)

print("Model path:", MODEL_PATH)
print("Model exists:", os.path.exists(MODEL_PATH))

cnn_model = tf.keras.models.load_model(MODEL_PATH)

print("CNN model loaded successfully")
print("Input shape:", cnn_model.input_shape)
print("Output shape:", cnn_model.output_shape)

Model path: /content/drive/MyDrive/EEG_Drowsiness_New/Models/final_1d_cnn.keras
Model exists: True
CNN model loaded successfully
Input shape: (None, 384, 30)
Output shape: (None, 1)


In [ ]:
TFLITE_PATH = os.path.join(
    MODEL_DIR,
    "final_1d_cnn.tflite"
)

converter = tf.lite.TFLiteConverter.from_keras_model(
    cnn_model
)

tflite_model = converter.convert()

with open(TFLITE_PATH, "wb") as file:
    file.write(tflite_model)

print("TensorFlow Lite model saved successfully")
print("Path:", TFLITE_PATH)
print(
    "File size:",
    os.path.getsize(TFLITE_PATH),
    "bytes"
)

Saved artifact at '/tmp/tmp67__c0mx'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 384, 30), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  136507976433936: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136507976444880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136507976437392: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136507976442000: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136507976443344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136507976443920: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136507976442192: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136507976437968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136507976444304: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136507976441424: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136507976434896: Te

In [ ]:
print("TFLite file exists:", os.path.exists(TFLITE_PATH))

if os.path.exists(TFLITE_PATH):
    print(
        "TFLite file size:",
        os.path.getsize(TFLITE_PATH),
        "bytes"
    )

TFLite file exists: True
TFLite file size: 66316 bytes


In [ ]:
import numpy as np

interpreter = tf.lite.Interpreter(
    model_path=TFLITE_PATH
)

interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("Input details:")
print(input_details)

print("\nOutput details:")
print(output_details)

Input details:
[{'name': 'serving_default_input_layer:0', 'index': 0, 'shape': array([  1, 384,  30], dtype=int32), 'shape_signature': array([ -1, 384,  30], dtype=int32), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0, 'block_size': 0}, 'sparsity_parameters': {}}]

Output details:
[{'name': 'StatefulPartitionedCall_1:0', 'index': 36, 'shape': array([1, 1], dtype=int32), 'shape_signature': array([-1,  1], dtype=int32), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0, 'block_size': 0}, 'sparsity_parameters': {}}]


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [ ]:
import os
import numpy as np

MODEL_DIR = (
    "/content/drive/MyDrive/"
    "EEG_Drowsiness_New/Models"
)

PREPROCESSOR_PATH = os.path.join(
    MODEL_DIR,
    "final_preprocessor.npz"
)

# Confirm that the raw EEG data is available
if "X" not in globals():
    raise NameError(
        "X is not defined. Run the dataset-loading cells first."
    )

print("Raw X shape:", X.shape)
print("Raw X dtype:", X.dtype)

# Load the saved preprocessing parameters
preprocessor = np.load(PREPROCESSOR_PATH)

lower = preprocessor["lower"].astype("float32")
upper = preprocessor["upper"].astype("float32")
mean = preprocessor["mean"].astype("float32")
std = preprocessor["std"].astype("float32")

# Apply the same preprocessing used during training
X_processed = np.clip(
    X,
    lower,
    upper
)

X_processed = (
    X_processed - mean
) / (
    std + 1e-8
)

X_processed = X_processed.astype("float32")

print("Processed X shape:", X_processed.shape)
print("Processed X dtype:", X_processed.dtype)
print("Processed minimum:", X_processed.min())
print("Processed maximum:", X_processed.max())
print("Processed mean:", X_processed.mean())
print("Processed standard deviation:", X_processed.std())

Raw X shape: (2022, 384, 30)
Raw X dtype: float32
Processed X shape: (2022, 384, 30)
Processed X dtype: float32
Processed minimum: -3.9797916
Processed maximum: 3.8623853
Processed mean: 4.058147e-10
Processed standard deviation: 0.9999996


In [ ]:
# Prepare one already-preprocessed EEG sample
test_sample = X_processed[0:1].astype(
    input_details[0]["dtype"]
)

interpreter.set_tensor(
    input_details[0]["index"],
    test_sample
)

interpreter.invoke()

tflite_output = interpreter.get_tensor(
    output_details[0]["index"]
)

tflite_probability = float(
    np.asarray(tflite_output).reshape(-1)[0]
)

tflite_prediction = int(
    tflite_probability >= 0.5
)

print("Test sample shape:", test_sample.shape)
print("TFLite probability:", tflite_probability)
print("TFLite prediction:", tflite_prediction)
print("True label:", int(y[0]))

Test sample shape: (1, 384, 30)
TFLite probability: 0.039652206003665924
TFLite prediction: 0
True label: 0


In [ ]:
import json

THRESHOLD_PATH = os.path.join(
    MODEL_DIR,
    "classification_thresholds.json"
)

with open(THRESHOLD_PATH, "r") as file:
    threshold_data = json.load(file)

CNN_THRESHOLD = float(
    threshold_data["cnn_threshold"]
)

print("Loaded CNN threshold:", CNN_THRESHOLD)

Loaded CNN threshold: 0.5


In [ ]:
tflite_prediction = int(
    tflite_probability >= CNN_THRESHOLD
)

print("Probability:", tflite_probability)
print("Threshold:", CNN_THRESHOLD)
print("Prediction:", tflite_prediction)
print("True label:", int(y[0]))

Probability: 0.039652206003665924
Threshold: 0.5
Prediction: 0
True label: 0


In [ ]:
required_files = [
    "final_1d_cnn.keras",
    "final_1d_cnn.tflite",
    "final_preprocessor.npz",
    "classification_thresholds.json"
]

print("Final deployment files:")

for filename in required_files:
    file_path = os.path.join(
        MODEL_DIR,
        filename
    )

    print(
        filename,
        "->",
        "OK" if os.path.exists(file_path) else "MISSING"
    )

Final deployment files:
final_1d_cnn.keras -> OK
final_1d_cnn.tflite -> OK
final_preprocessor.npz -> OK
classification_thresholds.json -> OK


In [ ]:
import os
import zipfile

deployment_files = [
    "final_1d_cnn.tflite",
    "final_preprocessor.npz",
    "classification_thresholds.json"
]

ZIP_PATH = os.path.join(
    MODEL_DIR,
    "raspberry_pi_deployment.zip"
)

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    zipfile.ZIP_DEFLATED
) as zip_file:

    for filename in deployment_files:
        file_path = os.path.join(
            MODEL_DIR,
            filename
        )

        if not os.path.exists(file_path):
            raise FileNotFoundError(
                f"Missing file: {file_path}"
            )

        zip_file.write(
            file_path,
            arcname=filename
        )

print("Deployment ZIP created:")
print(ZIP_PATH)
print(
    "ZIP size:",
    os.path.getsize(ZIP_PATH),
    "bytes"
)

Deployment ZIP created:
/content/drive/MyDrive/EEG_Drowsiness_New/Models/raspberry_pi_deployment.zip
ZIP size: 56863 bytes


In [ ]:
from google.colab import files

files.download(ZIP_PATH)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# FINAL PREPROCESSING FOR MULTI-SCALE CNN
# ============================================================

# 1. Calculate clipping limits from the complete dataset
lower = np.percentile(
    X, 1.0,
    axis=(0, 1),
    keepdims=True
)

upper = np.percentile(
    X, 99.0,
    axis=(0, 1),
    keepdims=True
)

# 2. Clip extreme values
X_clipped = np.clip(X, lower, upper)

# 3. Calculate normalization statistics
mean = X_clipped.mean(
    axis=(0, 1),
    keepdims=True
)

std = X_clipped.std(
    axis=(0, 1),
    keepdims=True
)

# Prevent division by zero
std = np.where(std < 1e-6, 1.0, std)

# 4. Normalize
X_final = (X_clipped - mean) / std

# 5. Convert to float32
X_final = X_final.astype(np.float32)

print("Final preprocessing completed")
print("--------------------------------")
print("Original shape :", X.shape)
print("Processed shape:", X_final.shape)
print("Data type      :", X_final.dtype)

print("\nStatistics after preprocessing:")
print("Minimum :", X_final.min())
print("Maximum :", X_final.max())
print("Mean    :", X_final.mean())
print("Std     :", X_final.std())

print("\nNaN values :", np.isnan(X_final).sum())
print("Inf values :", np.isinf(X_final).sum())

Final preprocessing completed
--------------------------------
Original shape : (2022, 384, 30)
Processed shape: (2022, 384, 30)
Data type      : float32

Statistics after preprocessing:
Minimum : -3.9797916
Maximum : 3.8623853
Mean    : 4.058147e-10
Std     : 0.9999996

NaN values : 0
Inf values : 0


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model

def build_multiscale_cnn(input_shape=(384, 30)):

    inputs = layers.Input(
        shape=input_shape,
        name="input_layer"
    )

    # ------------------------------------------------
    # Multi-scale branches
    # ------------------------------------------------
    branch_outputs = []

    for kernel_size in [3, 5, 7]:

        branch = layers.Conv1D(
            filters=32,
            kernel_size=kernel_size,
            padding="same",
            use_bias=False,
            name=f"conv_kernel_{kernel_size}"
        )(inputs)

        branch = layers.BatchNormalization(
            name=f"batch_norm_{kernel_size}"
        )(branch)

        branch = layers.Activation(
            "relu",
            name=f"relu_{kernel_size}"
        )(branch)

        branch = layers.MaxPooling1D(
            pool_size=2,
            name=f"pool_{kernel_size}"
        )(branch)

        branch_outputs.append(branch)

    # ------------------------------------------------
    # Combine multi-scale features
    # ------------------------------------------------
    x = layers.Concatenate(
        name="multiscale_concatenate"
    )(branch_outputs)

    # ------------------------------------------------
    # Feature fusion
    # ------------------------------------------------
    x = layers.Conv1D(
        filters=64,
        kernel_size=3,
        padding="same",
        use_bias=False,
        name="fusion_conv"
    )(x)

    x = layers.BatchNormalization(
        name="fusion_batch_norm"
    )(x)

    x = layers.Activation(
        "relu",
        name="fusion_relu"
    )(x)

    # ------------------------------------------------
    # Classification
    # ------------------------------------------------
    x = layers.GlobalAveragePooling1D(
        name="global_average_pooling"
    )(x)

    x = layers.Dense(
        32,
        activation="relu",
        name="dense_32"
    )(x)

    x = layers.Dropout(
        0.35,
        name="dropout"
    )(x)

    outputs = layers.Dense(
        1,
        activation="sigmoid",
        name="drowsiness_output"
    )(x)

    return Model(
        inputs=inputs,
        outputs=outputs,
        name="MultiScale_1D_CNN"
    )


# Build model
final_model = build_multiscale_cnn(
    input_shape=(384, 30)
)

# Display architecture
final_model.summary()

Model: "MultiScale_1D_CNN"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 384, 30)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_kernel_3       │ (None, 384, 32)   │      2,880 │ input_layer[0][0] │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_kernel_5       │ (None, 384, 32)   │      4,800 │ input_layer[0][0] │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_kernel_7       │ (None, 384, 32)   │      6,720 │ input_layer[0][0] │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_norm_3        │ (None, 384, 32)   │        128 │ conv_kernel_3[0]… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_norm_5        │ (None, 384, 32)   │        128 │ conv_kernel_5[0]… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_norm_7        │ (None, 384, 32)   │        128 │ conv_kernel_7[0]… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ relu_3 (Activation) │ (None, 384, 32)   │          0 │ batch_norm_3[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ relu_5 (Activation) │ (None, 384, 32)   │          0 │ batch_norm_5[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ relu_7 (Activation) │ (None, 384, 32)   │          0 │ batch_norm_7[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool_3              │ (None, 192, 32)   │          0 │ relu_3[0][0]      │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool_5              │ (None, 192, 32)   │          0 │ relu_5[0][0]      │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool_7              │ (None, 192, 32)   │          0 │ relu_7[0][0]      │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiscale_concate… │ (None, 192, 96)   │          0 │ pool_3[0][0],     │
│ (Concatenate)       │                   │            │ pool_5[0][0],     │
│                     │                   │            │ pool_7[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fusion_conv         │ (None, 192, 64)   │     18,432 │ multiscale_conca… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fusion_batch_norm   │ (None, 192, 64)   │        256 │ fusion_conv[0][0] │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fusion_relu         │ (None, 192, 64)   │          0 │ fusion_batch_nor… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 35,585 (139.00 KB)

 Trainable params: 35,265 (137.75 KB)

 Non-trainable params: 320 (1.25 KB)

In [ ]:
# ============================================================
# COMPILE FINAL MULTI-SCALE CNN
# ============================================================

final_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4,
        clipnorm=1.0
    ),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.AUC(name="auc")
    ]
)

print("Final Multi-scale CNN compiled successfully.")

print("\nOptimizer : Adam")
print("Learning rate : 1e-4")
print("Loss : Binary Crossentropy")
print("Metrics : Accuracy, AUC")

Final Multi-scale CNN compiled successfully.

Optimizer : Adam
Learning rate : 1e-4
Loss : Binary Crossentropy
Metrics : Accuracy, AUC


In [ ]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

# ============================================================
# FINAL TRAIN / VALIDATION SPLIT
# ============================================================

X_train, X_val, y_train, y_val = train_test_split(
    X_final,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Final training split")
print("--------------------")
print("Training samples  :", X_train.shape[0])
print("Validation samples:", X_val.shape[0])

print("\nTraining class distribution:")
print(np.bincount(y_train))

print("\nValidation class distribution:")
print(np.bincount(y_val))


# ============================================================
# CALLBACKS
# ============================================================

checkpoint_path = "/content/best_multiscale_cnn.keras"

early_stopping = EarlyStopping(
    monitor="val_auc",
    mode="max",
    patience=8,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_auc",
    mode="max",
    factor=0.5,
    patience=3,
    min_lr=1e-7,
    verbose=1
)

model_checkpoint = ModelCheckpoint(
    checkpoint_path,
    monitor="val_auc",
    mode="max",
    save_best_only=True,
    verbose=1
)


# ============================================================
# FINAL TRAINING
# ============================================================

history = final_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=40,
    batch_size=32,
    callbacks=[
        early_stopping,
        reduce_lr,
        model_checkpoint
    ],
    verbose=1
)

print("\n===================================")
print("FINAL MULTI-SCALE CNN TRAINING DONE")
print("===================================")
print("Best model saved to:")
print(checkpoint_path)

Final training split
--------------------
Training samples  : 1617
Validation samples: 405

Training class distribution:
[808 809]

Validation class distribution:
[203 202]
Epoch 1/40
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 0.4693 - auc: 0.4356 - loss: 0.8006
Epoch 1: val_auc improved from None to 0.79224, saving model to /content/best_multiscale_cnn.keras

Epoch 1: finished saving model to /content/best_multiscale_cnn.keras
51/51 ━━━━━━━━━━━━━━━━━━━━ 6s 57ms/step - accuracy: 0.5015 - auc: 0.4928 - loss: 0.7529 - val_accuracy: 0.5012 - val_auc: 0.7922 - val_loss: 0.6829 - learning_rate: 1.0000e-04
Epoch 2/40
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.5642 - auc: 0.6627 - loss: 0.6672
Epoch 2: val_auc improved from 0.79224 to 0.80244, saving model to /content/best_multiscale_cnn.keras

Epoch 2: finished saving model to /content/best_multiscale_cnn.keras
51/51 ━━━━━━━━━━━━━━━━━━━━ 2s 46ms/step - accuracy: 0.6073 - auc: 0.6832 - loss: 0.6502 - val_accuracy: 0.4988 -

In [ ]:
import os
import shutil

MODEL_DIR = "/content/drive/MyDrive/EEG_Drowsiness_New/Models"

os.makedirs(MODEL_DIR, exist_ok=True)

final_model_path = os.path.join(
    MODEL_DIR,
    "final_multiscale_cnn.keras"
)

shutil.copy(
    "/content/best_multiscale_cnn.keras",
    final_model_path
)

print("Final Multi-scale CNN saved successfully!")
print(final_model_path)
print("File size:", os.path.getsize(final_model_path), "bytes")

Final Multi-scale CNN saved successfully!
/content/drive/MyDrive/EEG_Drowsiness_New/Models/final_multiscale_cnn.keras
File size: 513965 bytes


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)
import numpy as np

# Load the best saved model
best_model = tf.keras.models.load_model(
    "/content/drive/MyDrive/EEG_Drowsiness_New/Models/final_multiscale_cnn.keras"
)

# Predict validation data
val_prob = best_model.predict(
    X_val,
    batch_size=32,
    verbose=1
).ravel()

# Default classification threshold
threshold = 0.50

val_pred = (val_prob >= threshold).astype(int)

# Metrics
accuracy = accuracy_score(y_val, val_pred)
balanced_acc = balanced_accuracy_score(y_val, val_pred)
precision = precision_score(y_val, val_pred, zero_division=0)
recall = recall_score(y_val, val_pred, zero_division=0)
f1 = f1_score(y_val, val_pred, zero_division=0)
auc = roc_auc_score(y_val, val_prob)

cm = confusion_matrix(y_val, val_pred)

print("\n======================================")
print("FINAL MULTI-SCALE CNN EVALUATION")
print("======================================")

print(f"Accuracy           : {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Balanced Accuracy  : {balanced_acc:.4f} ({balanced_acc*100:.2f}%)")
print(f"Precision          : {precision:.4f}")
print(f"Recall             : {recall:.4f}")
print(f"F1-score           : {f1:.4f}")
print(f"ROC-AUC            : {auc:.4f}")

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(
    classification_report(
        y_val,
        val_pred,
        target_names=["Alert", "Drowsy"],
        digits=4
    )
)

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step

FINAL MULTI-SCALE CNN EVALUATION
Accuracy           : 0.8395 (83.95%)
Balanced Accuracy  : 0.8393 (83.93%)
Precision          : 0.8914
Recall             : 0.7723
F1-score           : 0.8276
ROC-AUC            : 0.9452

Confusion Matrix:
[[184  19]
 [ 46 156]]

Classification Report:
              precision    recall  f1-score   support

       Alert     0.8000    0.9064    0.8499       203
      Drowsy     0.8914    0.7723    0.8276       202

    accuracy                         0.8395       405
   macro avg     0.8457    0.8393    0.8387       405
weighted avg     0.8456    0.8395    0.8388       405



In [ ]:
import tensorflow as tf
import os

# ============================================================
# CONVERT FINAL MULTI-SCALE CNN → TFLITE
# ============================================================

KERAS_MODEL_PATH = (
    "/content/drive/MyDrive/"
    "EEG_Drowsiness_New/Models/final_multiscale_cnn.keras"
)

TFLITE_MODEL_PATH = (
    "/content/drive/MyDrive/"
    "EEG_Drowsiness_New/Models/final_multiscale_cnn.tflite"
)

# Load final Keras model
model = tf.keras.models.load_model(KERAS_MODEL_PATH)

print("Keras model loaded successfully.")
print("Input shape :", model.input_shape)
print("Output shape:", model.output_shape)

# TFLite converter
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# Convert
tflite_model = converter.convert()

# Save
with open(TFLITE_MODEL_PATH, "wb") as f:
    f.write(tflite_model)

print("\n======================================")
print("TFLITE CONVERSION SUCCESSFUL")
print("======================================")
print("Saved to:")
print(TFLITE_MODEL_PATH)
print("File size:", os.path.getsize(TFLITE_MODEL_PATH), "bytes")

Keras model loaded successfully.
Input shape : (None, 384, 30)
Output shape: (None, 1)
Saved artifact at '/tmp/tmpom9hrgen'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 384, 30), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  133353861707920: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133353861707536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133353861707728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133353861706768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133353861706384: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133353861707152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133353861706960: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133353861705232: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133353861705424: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13

In [ ]:
import os
import shutil
import zipfile

MODEL_DIR = "/content/drive/MyDrive/EEG_Drowsiness_New/Models"

TFLITE_PATH = os.path.join(
    MODEL_DIR, "final_multiscale_cnn.tflite"
)

PREPROCESSOR_PATH = os.path.join(
    MODEL_DIR, "final_preprocessor.npz"
)

THRESHOLD_PATH = os.path.join(
    MODEL_DIR, "classification_thresholds.json"
)

# Create deployment folder
DEPLOY_DIR = "/content/multiscale_raspberry_pi_deployment"

os.makedirs(DEPLOY_DIR, exist_ok=True)

# Copy deployment files
shutil.copy(TFLITE_PATH, DEPLOY_DIR)
shutil.copy(PREPROCESSOR_PATH, DEPLOY_DIR)
shutil.copy(THRESHOLD_PATH, DEPLOY_DIR)

# Create ZIP
ZIP_PATH = os.path.join(
    MODEL_DIR,
    "multiscale_raspberry_pi_deployment.zip"
)

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    zipfile.ZIP_DEFLATED
) as zipf:

    for filename in os.listdir(DEPLOY_DIR):
        filepath = os.path.join(DEPLOY_DIR, filename)
        zipf.write(
            filepath,
            arcname=filename
        )

print("======================================")
print("RASPBERRY PI PACKAGE CREATED")
print("======================================")

print("\nFiles included:")

for filename in os.listdir(DEPLOY_DIR):
    filepath = os.path.join(DEPLOY_DIR, filename)
    print(f"{filename} : {os.path.getsize(filepath)} bytes")

print("\nZIP file:")
print(ZIP_PATH)

print("\nZIP size:", os.path.getsize(ZIP_PATH), "bytes")

RASPBERRY PI PACKAGE CREATED

Files included:
final_multiscale_cnn.tflite : 151740 bytes
classification_thresholds.json : 28 bytes
final_preprocessor.npz : 1464 bytes

ZIP file:
/content/drive/MyDrive/EEG_Drowsiness_New/Models/multiscale_raspberry_pi_deployment.zip

ZIP size: 134343 bytes
